Damped Newton method 
chi=30, truncating to 14,14 per leg
Start Newton method after 4 RG steps from T=T_c.
Damping with newton_step=0.5 is activated a couple of times (e.g. for i=3)
Here I ran up to i=9, reaching fp_error =2.3e-5. 

Here 20 eigenvalues are used (for no particular reason, I guess 10 eigenvalues would work as well). 

gilt_eps=2e-5

Compared to the previous version of the code, discrete gauge fixing when computing the Jacobian is carried out using the gauge-fixing element set of R(A), not of A. This eliminates a lot of warnings.

In [52]:
using Pkg
Pkg.activate(".")
include("Tools.jl")
include("KrylovTechnical.jl")
include("GaugeFixing.jl");
include("./Lab/newton-step-SR.jl");

  Activating project at `~/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R`
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """
GiltTNR/GiltTNR2D_essentials.py:113: SyntaxWarning: invalid escape sequence '\ '
  """


In [47]:
gilt_eps = 6e-6
chi = 30
trunc_shape = [14 14; 14 14; 14 14; 14 14]  # shape to truncate to, not to deal with Gilt tensor dimension oscillations
cg_eps = 1e-10
newton_eps = 1e-9
gilt_pars = Dict(
	"gilt_eps" => gilt_eps,
	"cg_chis" => collect(1:chi),
	"cg_eps" => cg_eps,
	"verbosity" => 1,
	"rotate" => true,
    "bond_repetitions" =>2,
    "recursion_depth" => Dict(
		"S" => 80,
		"N" => 80,
		"E" => 80,
		"W" => 80,
        )
)
Jratio = 1.0

relT=1.0
rg_steps = 10
#do rg_steps steps from the critical tensor
initialA_pars = Dict("relT" => relT, "Jratio" => Jratio)
traj = trajectory(initialA_pars, rg_steps, gilt_pars, fix_cont_gauge = true, verbose = true)["A"];
#NB traj consists of PyObjects

#traj = traj .|> x -> fix_continuous_gauge(x)[1]; #this is still PyObjects
#traj[rg_steps+1], accepted_elements, _ = fix_discrete_gauge(traj[rg_steps+1]; tol = 1e-7);

function fix_discrete_by_accepted_elements_if_possible(x)
	res = x
	try
		res = fix_discrete_gauge(x, accepted_elements)[1]
	catch
		res = fix_discrete_gauge(x)[1]
	end
	return res
end

#traj = traj .|> x -> fix_discrete_by_accepted_elements_if_possible(x);
traj = py_to_ju.(traj);
traj = traj .|> x -> x / norm(x); 

for i in 1:length(traj)
    println(i," ",traj[i].shape, traj[i].qhape )
end

/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return sinh(2*x*Jv)*sinh(2*x*Jh) - 1
/Users/slava/Personal/----Fisica----/_Projects/Fork/GILT_TNR_R/GiltTNR/GiltTNR2D_Ising_benchmarks.py:169: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array 

1 [1 1; 1 1; 1 1; 1 1][0 1; 0 1; 0 1; 0 1]
2 [2 2; 2 2; 2 2; 2 2][1 0; 1 0; 1 0; 1 0]
3 [8 8; 8 8; 8 8; 8 8][1 0; 1 0; 1 0; 1 0]
4 [16 14; 16 14; 16 14; 16 14][1 0; 1 0; 1 0; 1 0]
5 [15 15; 15 15; 15 15; 15 15][1 0; 1 0; 1 0; 1 0]
6 [15 15; 15 15; 15 15; 15 15][1 0; 1 0; 1 0; 1 0]
7 [15 15; 16 14; 15 15; 16 14][1 0; 1 0; 1 0; 1 0]
8 [16 14; 15 15; 16 14; 15 15][1 0; 1 0; 1 0; 1 0]
9 [15 15; 15 15; 15 15; 15 15][1 0; 1 0; 1 0; 1 0]
10 [15 15; 15 15; 15 15; 15 15][1 0; 1 0; 1 0; 1 0]
11 [15 15; 15 15; 15 15; 15 15][1 0; 1 0; 1 0; 1 0]


In [53]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method


for i in 4:20
    A[i+1] = truncate_blocks(traj[i+1], trunc_shape)
    A[i] = truncate_blocks(traj[i], trunc_shape)
    A[i+1], accepted_elements[i] = fix_discrete_gauge(A[i+1]; tol = 1e-7);
    
    A[i], _ = fix_discrete_gauge(ju_to_py(A[i]), accepted_elements[i])
    A[i] = py_to_ju(A[i])
    e0 = embedded_distance(A[i+1], A[i])
    println("i= $i , ||R(A[i])-A[i]||= $e0 ")
end
    

i= 4 , ||R(A[i])-A[i]||= 0.1217783817736247 
i= 5 , ||R(A[i])-A[i]||= 0.06583115807255865 
i= 6 , ||R(A[i])-A[i]||= 0.06408293168985131 
i= 7 , ||R(A[i])-A[i]||= 0.05629616443743301 
i= 8 , ||R(A[i])-A[i]||= 0.0790057272296516 
i= 9 , ||R(A[i])-A[i]||= 0.04242093866253127 
i= 10 , ||R(A[i])-A[i]||= 0.10337078856755745 


LoadError: BoundsError: attempt to access 11-element Vector{Z2Tensor} at index [12]

In [49]:
for i in 4:20
    A[i+1] = truncate_blocks(traj[i+1], trunc_shape)
    A[i] = truncate_blocks(traj[i], trunc_shape)
    A[i+1] = abs.(ju_to_py(A[i+1]).to_ndarray())
    A[i] = abs.(ju_to_py(A[i]).to_ndarray())
    
    e0 = norm(A[i+1]-A[i])
    println("i= $i , ||R(A[i])-A[i]||= $e0 ")
end

i= 4 , ||R(A[i])-A[i]||= 0.12110113436686035 
i= 5 , ||R(A[i])-A[i]||= 0.06549734925925234 
i= 6 , ||R(A[i])-A[i]||= 0.06357557181958452 
i= 7 , ||R(A[i])-A[i]||= 0.05603884838257567 
i= 8 , ||R(A[i])-A[i]||= 0.07571003556309987 
i= 9 , ||R(A[i])-A[i]||= 0.040974816632172464 
i= 10 , ||R(A[i])-A[i]||= 0.08567982694545992 


LoadError: BoundsError: attempt to access 11-element Vector{Z2Tensor} at index [12]

In [55]:
A = Any[ NaN for _ in 1:40 ]; # list of tensors, Newton method trajectory
accepted_elements = Any[ NaN for _ in 1:40 ]; # list of elements in gauge-fixing
deltaA = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method
deltaA2 = Any[ NaN for _ in 1:40 ]; # list of deltaA's proposed by Newton method

A[1] = truncate_blocks(traj[7], trunc_shape)
for i in 1:30
    println("i=",i)

    RA = gilt_with_cont_gauge(A[i], gilt_pars; trunc_shape = trunc_shape);
    RA, accepted_elements[i] = fix_discrete_gauge(RA; tol = 1e-7);
    
    A[i], _ = fix_discrete_gauge(ju_to_py(A[i]), accepted_elements[i])
    A[i] = py_to_ju(A[i])

    e0 = embedded_distance(RA, A[i])
    println("||R(A[i])-A[i]||= ", e0)
    println("shapes:", A[i].shape, RA.shape)
    flush(stdout)
    
    deltaA[i],  deltaA2[i] = newton_correction_with_iterations_fixed(A[i], 20, accepted_elements[i], 
        gilt_pars; trunc_shape = trunc_shape, gmres = false, recompute_gilt_pars = false, order = 2);
    
    println("||deltaA[i]||,||deltaA2[i]|| = ", norm(deltaA[i]), " ",norm(deltaA2[i]))

    function phi(a)
        A0 = A[i] + a * deltaA[i]
        
        A0, _ = fix_discrete_gauge(ju_to_py(A0), accepted_elements[i])
        A0 = py_to_ju(A0)
        
        RA = gilt_with_cont_gauge(A0, gilt_pars; trunc_shape = trunc_shape);
        RA, _ = fix_discrete_gauge(ju_to_py(RA), accepted_elements[i]);
        RA = py_to_ju(RA)
    
        return embedded_distance(RA, A0)
    end
    
    #println(phi.(0.1:0.1:1.5))

    #throw("")
    
    newton_step = 1.0
    enew = e0
    while true #damped Newton method implementation, which reduces a step by 2 if cost function does not decrease
        println("newton_step= ", newton_step)
        Anew = A[i] + newton_step * (deltaA[i] + deltaA2[i])

        RAnew = gilt_with_cont_gauge(Anew, gilt_pars; trunc_shape = trunc_shape);
        RAnew, accepted_elements_new = fix_discrete_gauge(RAnew; tol = 1e-7);
    
        Anew, _ = fix_discrete_gauge(ju_to_py(Anew), accepted_elements_new)
        Anew = py_to_ju(Anew)

        enew = embedded_distance(RAnew, Anew)
        
        println("fp_error= ", enew)
        println("shapes:", Anew.shape, RAnew.shape)
        if enew < e0 && Anew.shape == RAnew.shape
            A[i+1] = Anew
            break
        end
        newton_step *= 0.5 
    end
    if enew < newton_eps
        break
    end
end

i=1
||R(A[i])-A[i]||= 0.05644245948801394
shapes:[14 14; 14 14; 14 14; 14 14][14 14; 14 14; 14 14; 14 14]
EIGENVALUES (INITIAL):
1.9891888573921503 + 0.0im
-0.951807476375966 + 0.0im
-0.9396929919434831 + 0.0im
0.7832040976663203 + 0.0im
0.7147027623746243 + 0.0im
0.09659255422204735 + 0.5367386255470518im
0.09659255422204735 - 0.5367386255470518im
0.47772876888905175 + 0.0im
0.4011339778382533 + 0.0im
-0.3182464856928747 + 0.1327335070787405im
-0.3182464856928747 - 0.1327335070787405im
-0.05098471806170382 + 0.33476005355635086im
-0.05098471806170382 - 0.33476005355635086im
0.09655822840902162 + 0.2938075762902037im
0.09655822840902162 - 0.2938075762902037im
-0.07818326182801935 + 0.29298783420395214im
-0.07818326182801935 - 0.29298783420395214im
0.029032413044082262 + 0.2829042456124843im
0.029032413044082262 - 0.2829042456124843im
-0.2682773684690031 + 0.0im
||deltaA[i]||,||deltaA2[i]|| = 0.3030012237994397 0.02373856077420735
newton_step= 1.0
fp_error= 0.32136280938256623
shapes:[1

┌ Info: Arnoldi eigsolve finished after 3 iterations:
│ *  20 eigenvalues converged
│ *  norm of residuals = (2.690388659076142e-54, 3.840807593417104e-40, 4.218638339779192e-40, 1.6914162493040586e-37, 6.737847128467165e-35, 1.2216497227486223e-29, 1.2216497227486223e-29, 5.848879651806576e-24, 5.261239888382829e-19, 1.09014335699846e-20, 1.09014335699846e-20, 2.5966093411944028e-18, 2.5966093411944028e-18, 1.0186069122936669e-16, 1.0186069122936669e-16, 3.0823382695893902e-15, 3.0823382695893902e-15, 7.064978004965824e-15, 7.064978004965824e-15, 3.203676558542998e-15)
└ *  number of operations = 64
┌ Info: Arnoldi eigsolve finished after 4 iterations:
│ *  20 eigenvalues converged
│ *  norm of residuals = (4.2537647383791003e-60, 1.0149333214710556e-59, 1.0149333214710556e-59, 9.520360938128577e-47, 1.1293573010495819e-44, 2.128723502519319e-39, 2.2581143007606415e-31, 2.2581143007606415e-31, 6.070091590807871e-30, 5.838831707395583e-29, 2.0585698514139154e-25, 2.0585698514139154e-25

LoadError: InterruptException: